# Stage 3: features + XGBoost + the same HMM

Same 16 features as the forest in Stage 1, but a different kind of tree model. Everything else is the same: people, HMM and metrics.

**Random forest vs XGBoost.** A forest grows many deep trees *independently* and lets them vote. XGBoost grows small trees *one after another*, each one fixing the mistakes the trees before it still make (this is called *boosting*). On tables of features it is often a little more accurate, and its model can be much smaller.

1. Same people and features as Stage 1
2. Choose the settings, using training people only, on the GPU
3. Train, then smooth with the HMM from Stage 1
4. Score on the test people, next to the forest and CNN

**Training runs on the RTX 3060 only, never the CPU.** If the GPU is missing, the notebook stops.

**Kernel:** the `.venv-tf` environment in this folder (XGBoost 3.2). This stage was added after Stage 2, so the test people had already been seen once; the settings below are still chosen on training people only.

## How this works: XGBoost + HMM, explained from scratch

XGBoost uses the **same 16 features** as the forest in Stage 1, and decision trees too (see Stage 1 for what a tree is). The difference is **how the trees work together**.

### Forest vs boosting, as a story

- **Random forest:** 200 students each study a different random part of the material, *separately*. In the exam they vote.
- **XGBoost (boosting):** students study *one after another*. Each new student only studies the questions the class still gets wrong, and adds a small correction to the class's answer.

### Step by step

1. Start with a plain guess for every window (roughly: how common each class is)
2. **Tree 1** (small, at most 8 questions deep) learns a correction that makes the guesses less wrong
3. Its correction is shrunk to 10% (the **learning rate**, 0.1) and added. Small steps make it harder to overreact to noise
4. **Tree 2** looks at what is *still* wrong after tree 1, and learns a correction for that
5. ... repeat 200 times. The final answer = the plain guess + 200 small corrections, turned into 4 probabilities

Because each tree works on what's still wrong, the later trees spend their effort on the **hard windows**, often the rare moderate-vigorous ones and the borderline light ones.

*(XGBoost grows one small tree per class each round, so 200 rounds x 4 classes = 800 small trees.)*

### Step 4: decisions we made, and why

| Decision | Why |
|---|---|
| Same 16 features as Stage 1 (and checked to give identical numbers) | So the comparison is tree-model vs tree-model, nothing else changes |
| Tree depth {4, 6, 8} x trees {200, 500} tried with 5-fold validation grouped by person, on training people only | Same rule as before: settings are never chosen on the test people |
| Weights per window (`compute_sample_weight('balanced')`) | XGBoost has no `class_weight` option. This does the same job |
| Trained on the RTX 3060 (`device='cuda'`) | The rule is no training on the CPU. On the GPU it trains in seconds |
| The **same** HMM as Stages 1 and 2 | So any difference comes from the model, not the smoothing |

**Honesty note:** this stage was added after Stages 1 and 2 had been scored, so the test people had already been looked at once. Its settings still came from training people only.

### The HMM: fixing one window at a time with common sense

The model judges each 30 s window **on its own**. But people don't switch activity every 30 seconds. If the model says

`sleep, sleep, light, sleep, sleep`

at 3 am, that lone "light" is almost certainly a mistake (maybe the person rolled over).

The **HMM** (hidden Markov model) fixes this by looking at the whole day at once. It weighs two things:

1. **What the model says** about each window (its probabilities, e.g. "70% sleep, 25% light")
2. **How often people really switch** from one activity to another, counted from the training people's labels (in the training people, the next window after sleep is sleep again 99.9% of the time; after light it is light 95% of the time)

The **Viterbi** algorithm then finds the single most likely sequence of labels for the whole day. The lone "light" gets overruled, and you get `sleep, sleep, sleep, sleep, sleep`. A long run of "light" in the morning would be kept, because the evidence is strong for many windows in a row.

### How the results are scored

| Score | What it asks | Why we use it |
|---|---|---|
| **Accuracy** | What share of windows is right? | Easy to read, but flattering: always guessing "sedentary" already gets about 40% |
| **Macro-F1** | How good is it on *each* class, averaged with equal weight? | The rare moderate-vigorous class counts as much as sleep |
| **Cohen's kappa** | How much better than lucky guessing? 0 = guessing, 1 = perfect | The fairest single number when classes are uneven |

Each is reported for the **4 classes** and for **binary** (still = sleep + sedentary, moving = light + moderate-vigorous). Binary is the simple "is this person moving?" answer the watch needs most.

In [1]:
import time

import nbformat
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_sample_weight

from common import (BINARY, CLASSES, DATA_DIR, FEATURES, OLD_TEST_FILE, RESULTS_DIR, SEED, confusion_table, evaluate,
                    learn_hmm, make_features, scores, smooth, split_people, to_binary)

# WHY: the rule is GPU only. Stop here instead of quietly training on the CPU.
assert xgb.build_info()['USE_CUDA'], 'this XGBoost cannot use the GPU'
print('XGBoost', xgb.__version__)

XGBoost 3.2.0


## 1. Same people, same features

`make_features` now lives in `common.py`. The first check runs the original Stage 1 cell on two people and makes sure the copy gives exactly the same numbers.

In [2]:
train_ids, test_ids = split_people()
assert test_ids == pd.read_csv(RESULTS_DIR / 'test_participants.csv')['participant_id'].tolist()

# the feature code exactly as Stage 1 ran it
stage1_cells = [x.source for x in nbformat.read('stage1_rf_hmm.ipynb', 4).cells if x.cell_type == 'code']
stage1_code = stage1_cells[2].split('# check on fake')[0]
stage1 = {'np': np}
exec(stage1_code, stage1)
for p in [train_ids[0], test_ids[0]]:
    with np.load(DATA_DIR / f'{p}.npz') as f:
        assert np.array_equal(make_features(f['X']), stage1['make_features'](f['X'])), p
assert FEATURES == stage1['FEATURES']
print('features are identical to Stage 1')


def load_features(ids):
    """One person at a time -> features, labels, person id, start time (in time order within each person)."""
    F, y, pid, start = [], [], [], []
    for p in ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            F.append(make_features(f['X'])); y.append(f['y4']); start.append(f['start'])
        pid.append(np.full(len(y[-1]), p))
    return np.concatenate(F), np.concatenate(y).astype(int), np.concatenate(pid), np.concatenate(start)

t0 = time.time()
F_train, y_train, pid_train, _ = load_features(train_ids)
F_test, y_test, pid_test, start_test = load_features(test_ids)
rf_saved = np.load(RESULTS_DIR / 'stage1_predictions.npz')
# the same windows, in the same order, as the forest was scored on
assert (y_test == rf_saved['y']).all() and (pid_test == rf_saved['pid']).all() and (start_test == rf_saved['start']).all()
print(f'train {F_train.shape}, test {F_test.shape}, {time.time() - t0:.0f} s')

features are identical to Stage 1


train (243012, 16), test (58152, 16), 45 s


## 2. Choose the settings, using training people only

Two settings matter most:

- `max_depth`: how many questions each small tree may ask. Deeper = more detail, but more risk of memorising people.
- `n_estimators`: how many trees are added one after another.

The learning rate (how much each new tree is allowed to correct) is fixed at 0.1. Five-fold validation **grouped by person**, the same as the forest, and the best 4-class macro-F1 wins.

**Class weights:** XGBoost has no `class_weight='balanced'`, so each window gets a weight instead. Same effect: a rare moderate-vigorous window counts for more.

In [3]:
def make_xgb(depth, n_trees):
    return xgb.XGBClassifier(n_estimators=n_trees, max_depth=depth, learning_rate=0.1,
                             objective='multi:softprob', tree_method='hist',
                             device='cuda',                # train on the RTX 3060
                             n_jobs=4,                     # WHY: keep the CPU cool while it feeds the GPU
                             random_state=SEED)

def fit(model, F, y):
    return model.fit(F, y, sample_weight=compute_sample_weight('balanced', y))

# check that training really happens on the GPU
check = fit(make_xgb(4, 5), F_train[:5000], y_train[:5000])
assert '"device":"cuda' in check.get_booster().save_config().replace(' ', ''), 'not training on the GPU'
print('training on the GPU: yes')

rows = []
t0 = time.time()
for depth in [4, 6, 8]:
    for n_trees in [200, 500]:
        fold_f1 = []
        for fit_idx, val_idx in GroupKFold(5).split(F_train, y_train, pid_train):
            assert not set(pid_train[fit_idx]) & set(pid_train[val_idx])
            model = fit(make_xgb(depth, n_trees), F_train[fit_idx], y_train[fit_idx])
            fold_f1.append(scores(y_train[val_idx], model.predict(F_train[val_idx]), 4)['macro-F1'])
        rows.append({'max_depth': depth, 'n_estimators': n_trees,
                     'mean macro-F1': np.mean(fold_f1), 'std over folds': np.std(fold_f1)})
        print(rows[-1], f'{time.time() - t0:.0f} s')
tuning = pd.DataFrame(rows)
tuning.to_csv(RESULTS_DIR / 'stage3_xgb_tuning.csv', index=False)
best = tuning.loc[tuning['mean macro-F1'].idxmax()]
best_depth, best_trees = int(best['max_depth']), int(best['n_estimators'])
print(tuning.round(3).to_string(index=False))
print(f'chosen: max_depth {best_depth}, n_estimators {best_trees}. (The forest got 0.748 in the same kind of check.)')

training on the GPU: yes


C:\Users\syafi\SilverGuard Major Project\notebooks\DEEPDEEPDEEP_learning\.venv-tf\Lib\site-packages\xgboost\core.py:751: UserWarning: [14:37:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


{'max_depth': 4, 'n_estimators': 200, 'mean macro-F1': np.float64(0.7331500561857387), 'std over folds': np.float64(0.02047247081981801)} 9 s


{'max_depth': 4, 'n_estimators': 500, 'mean macro-F1': np.float64(0.7357720003701073), 'std over folds': np.float64(0.021169578662362885)} 31 s


{'max_depth': 6, 'n_estimators': 200, 'mean macro-F1': np.float64(0.738476403241893), 'std over folds': np.float64(0.020022351367361083)} 45 s


{'max_depth': 6, 'n_estimators': 500, 'mean macro-F1': np.float64(0.7359723994461181), 'std over folds': np.float64(0.020867816743563812)} 74 s


{'max_depth': 8, 'n_estimators': 200, 'mean macro-F1': np.float64(0.740126618300474), 'std over folds': np.float64(0.02019977814411413)} 97 s


{'max_depth': 8, 'n_estimators': 500, 'mean macro-F1': np.float64(0.7345836814468935), 'std over folds': np.float64(0.01841270512820526)} 150 s
 max_depth  n_estimators  mean macro-F1  std over folds
         4           200          0.733           0.020
         4           500          0.736           0.021
         6           200          0.738           0.020
         6           500          0.736           0.021
         8           200          0.740           0.020
         8           500          0.735           0.018
chosen: max_depth 8, n_estimators 200. (The forest got 0.748 in the same kind of check.)


## 3. Train on all training people, then the same HMM

In [4]:
t0 = time.time()
booster = fit(make_xgb(best_depth, best_trees), F_train, y_train)
print(f'trained in {time.time() - t0:.0f} s')

hmm = np.load(RESULTS_DIR / 'hmm.npz')
start_p, transition = learn_hmm(y_train, pid_train)
assert np.allclose(start_p, hmm['start']) and np.allclose(transition, hmm['transition'])   # truly the same HMM

probs = booster.predict_proba(F_test)
assert probs.shape == (len(y_test), 4) and np.allclose(probs.sum(axis=1), 1, atol=1e-4)
pred_xgb = probs.argmax(axis=1)
pred_hmm = smooth(probs, pid_test, hmm['start'], hmm['transition'])

trained in 5 s


## 4. Score on the test people, next to Stage 1 and Stage 2

In [5]:
stage3_table = pd.DataFrame(evaluate('XGB', y_test, pred_xgb) + evaluate('XGB+HMM', y_test, pred_hmm))
stage3_table.to_csv(RESULTS_DIR / 'stage3_xgb_metrics.csv', index=False)
table = pd.concat([pd.read_csv(RESULTS_DIR / 'stage1_metrics.csv'), pd.read_csv(RESULTS_DIR / 'stage2_metrics.csv'), stage3_table])
order = ['always sedentary', 'RF', 'RF+HMM', 'CNN', 'CNN+HMM', 'XGB', 'XGB+HMM']
print(table.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).loc[order].round(3).to_string())

                 accuracy        macro-F1          kappa       
task              4-class binary  4-class binary 4-class binary
model                                                          
always sedentary    0.414  0.809    0.146  0.447   0.000  0.000
RF                  0.806  0.907    0.741  0.852   0.701  0.705
RF+HMM              0.880  0.933    0.801  0.889   0.814  0.778
CNN                 0.776  0.905    0.710  0.852   0.659  0.705
CNN+HMM             0.848  0.923    0.784  0.879   0.768  0.758
XGB                 0.800  0.901    0.741  0.848   0.696  0.697
XGB+HMM             0.882  0.934    0.823  0.895   0.818  0.790


In [6]:
for name, pred in [('XGB', pred_xgb), ('XGB+HMM', pred_hmm)]:
    print(f'===== {name}: 4-class =====')
    print(confusion_table(y_test, pred, CLASSES).to_string())
    print(f'\n===== {name}: binary =====')
    print(confusion_table(to_binary(y_test), to_binary(pred), BINARY).to_string(), '\n')

===== XGB: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              19893       2687    346                 21     86.7         11.7      1.5                  0.1
true sedentary           1625      19119   3156                175      6.7         79.4     13.1                  0.7
true light                 65       1853   5725                796      0.8         22.0     67.8                  9.4
true moderate-vigorous      5        134    770               1782      0.2          5.0     28.6                 66.2

===== XGB: binary =====
             still  moving  still %  moving %
true still   43324    3698     92.1       7.9
true moving   2057    9073     18.5      81.5 

===== XGB+HMM: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              21694       1069    184                  0 

## 5. Person by person

The honest sample size is 30 people, so: for how many of them is XGBoost + HMM better?

In [7]:
cnn_saved = np.load(RESULTS_DIR / 'stage2_predictions.npz')
assert (cnn_saved['y'] == y_test).all() and (cnn_saved['pid'] == pid_test).all()
preds = {'RF+HMM': rf_saved['pred_hmm'], 'CNN+HMM': cnn_saved['pred_hmm'], 'XGB': pred_xgb, 'XGB+HMM': pred_hmm}
ages = pd.read_csv('../../data/open datasets/capture24/capture24/metadata.csv').set_index('pid')['age']
old_test = set(pd.read_csv(OLD_TEST_FILE)['participant_id'])

rows = []
for p in test_ids:
    m = pid_test == p
    row = {'pid': p, 'age': ages[p], 'old_test': p in old_test}
    for name, pred in preds.items():
        row[f'{name} 4-class'] = scores(y_test[m], pred[m], 4)['kappa']
        row[f'{name} binary'] = scores(to_binary(y_test[m]), to_binary(pred[m]), 2)['kappa']
    rows.append(row)
people = pd.DataFrame(rows)
people.to_csv(RESULTS_DIR / 'stage3_xgb_per_person.csv', index=False)

print('median kappa per person:')
print(pd.DataFrame({t: {n: people[f'{n} {t}'].median() for n in preds} for t in ['4-class', 'binary']}).round(3).to_string())
for a, b in [('XGB+HMM', 'RF+HMM'), ('XGB+HMM', 'CNN+HMM'), ('XGB+HMM', 'XGB')]:
    for t in ['4-class', 'binary']:
        d = people[f'{a} {t}'] - people[f'{b} {t}']
        print(f'{a} beats {b}, {t}: {(d > 0).sum()} / 30 people, median difference in kappa {d.median():+.3f}')
for label, mask in [('53+', people.age == '53+'), ('under 53', people.age != '53+')]:
    print(f'median 4-class kappa, {label} (n={mask.sum()}):',
          {n: round(float(people.loc[mask, f'{n} 4-class'].median()), 3) for n in preds})

median kappa per person:
         4-class  binary
RF+HMM     0.819   0.798
CNN+HMM    0.796   0.768
XGB        0.713   0.696
XGB+HMM    0.823   0.807
XGB+HMM beats RF+HMM, 4-class: 19 / 30 people, median difference in kappa +0.008
XGB+HMM beats RF+HMM, binary: 16 / 30 people, median difference in kappa +0.002
XGB+HMM beats CNN+HMM, 4-class: 25 / 30 people, median difference in kappa +0.047
XGB+HMM beats CNN+HMM, binary: 18 / 30 people, median difference in kappa +0.022
XGB+HMM beats XGB, 4-class: 30 / 30 people, median difference in kappa +0.120
XGB+HMM beats XGB, binary: 29 / 30 people, median difference in kappa +0.080
median 4-class kappa, 53+ (n=9): {'RF+HMM': 0.79, 'CNN+HMM': 0.689, 'XGB': 0.638, 'XGB+HMM': 0.798}
median 4-class kappa, under 53 (n=21): {'RF+HMM': 0.853, 'CNN+HMM': 0.829, 'XGB': 0.727, 'XGB+HMM': 0.832}


## 6. Save, and how big is each model?

Size matters for the watch. These are file sizes on the laptop, not yet the size on the ESP32 (that comes with export, later).

In [8]:
booster.save_model(RESULTS_DIR / 'xgb_4class.json')
np.savez(RESULTS_DIR / 'stage3_xgb_predictions.npz', pid=pid_test, start=start_test, y=y_test,
         probs=probs, pred_xgb=pred_xgb, pred_hmm=pred_hmm)

# the saved model gives the same answers as the one in memory
again = xgb.XGBClassifier()
again.load_model(RESULTS_DIR / 'xgb_4class.json')
assert np.allclose(again.predict_proba(F_test[:2000]), probs[:2000], atol=1e-5)

print(f'XGBoost: {best_trees} rounds x 4 classes = {best_trees * 4} small trees, depth up to {best_depth}')
for name in ['rf_4class.joblib', 'cnn_4class.keras', 'xgb_4class.json']:
    print(f'{name:20s} {(RESULTS_DIR / name).stat().st_size / 1e6:8.2f} MB')

XGBoost: 200 rounds x 4 classes = 800 small trees, depth up to 8
rf_4class.joblib       136.15 MB
cnn_4class.keras         0.50 MB
xgb_4class.json         14.92 MB
